# 채용 플랫폼 사용자 행동 기반 세그먼테이션 (RJA)

팀 프로젝트 **「공고는 봤는데, 왜 지원하지 않았을까?」** 중 사용자 세그먼테이션 부분만 별도로 정리했습니다.

- 분석 기준일: **2023-12-31**
- R: 최근 활동일 경과 / J: 공고 상세 조회 횟수 / A: 지원 횟수
- 원본 팀 노트북의 **점수 구간 및 4개 그룹 분류 기준**을 유지했습니다.
- CSV 원본과 사용자 식별자가 담긴 실행 출력은 공개하지 않습니다.

## 01. 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from urllib.parse import urlparse

# 원본 팀 노트북의 파일명 그대로 사용
application_df = pd.read_csv('applicaion_2023.csv')
log_df = pd.read_csv('log2023_df_url.csv', usecols=['user_uuid', 'URL', 'date', 'method'])

print('지원 데이터:', application_df.shape)
print('활동 로그:', log_df.shape)

## 02. 사용자별 최근 활동과 지원 횟수

In [ ]:
# 지원 내역의 사용자별 레코드 수
application_counts = application_df.groupby('user_uuid').size().rename('application_count')

log_df['date'] = pd.to_datetime(log_df['date'], errors='coerce')
if log_df['date'].isna().any():
    raise ValueError('date 컬럼에 날짜로 변환할 수 없는 값이 있습니다.')

rja_df = log_df.groupby('user_uuid')['date'].max().to_frame('last_active_date')
rja_df['Recency'] = (pd.Timestamp('2023-12-31') - rja_df['last_active_date']).dt.days

if not rja_df['Recency'].between(0, 365).all():
    raise ValueError('2023년 분석 기준일 밖의 날짜가 있습니다.')

rja_df = rja_df.join(application_counts).fillna({'application_count': 0})
rja_df['application_count'] = rja_df['application_count'].astype(int)

conversion_rate = rja_df['application_count'].gt(0).mean() * 100
print(f'전체 분석 사용자: {len(rja_df):,}명')
print(f'지원 경험 사용자 비율: {conversion_rate:.2f}%')

## 03. 공고 상세 조회 횟수 집계

In [ ]:
# 원본 분석의 상세 페이지 path / method 조건 유지
log_df['path'] = log_df['URL'].fillna('').apply(lambda url: urlparse(url).path)
job_views = (
    log_df.loc[
        log_df['path'].eq('api/jobs/job_title') &
        log_df['method'].eq('GET')
    ]
    .groupby('user_uuid').size().rename('job_view')
)

rja_df = rja_df.join(job_views).fillna({'job_view': 0})
rja_df['job_view'] = rja_df['job_view'].astype(int)

## 04. RJA 점수화

| 지표 | 1점 | 2점 | 3점 | 4점 |
| --- | --- | --- | --- | --- |
| R: 최근 활동 경과일 | 241~365일 | 119~240일 | 60~118일 | 0~59일 |
| J: 공고 상세 조회 횟수 | 0회 | 1~6회 | 7~31회 | 32회 이상 |
| A: 지원 횟수 | 0회 | 1~2회 | 3~8회 | 9회 이상 |

점수 구간은 원본 팀 프로젝트의 설정을 그대로 사용합니다.

In [ ]:
rja_df['R_score'] = pd.cut(
    rja_df['Recency'], bins=[-1, 59, 118, 240, 365], labels=[4, 3, 2, 1]
).astype(int)

rja_df['J_score'] = np.select(
    [rja_df['job_view'].eq(0), rja_df['job_view'].le(6), rja_df['job_view'].le(31)],
    [1, 2, 3], default=4
)

rja_df['A_score'] = np.select(
    [rja_df['application_count'].eq(0),
     rja_df['application_count'].le(2),
     rja_df['application_count'].le(8)],
    [1, 2, 3], default=4
)

## 05. 세그먼트 분류

원본 팀 분석의 분류 **우선순위**를 유지했습니다.

- **핵심 구직자:** R ≥ 3, J ≥ 3, A ≥ 3
- **적극 탐색 구직자:** R ≥ 3, J ≥ 3, A < 3
- **이탈 위험 구직자:** 위 두 그룹이 아니면서 R ≥ 2, J ≥ 2
- **휴면 구직자:** 그 밖의 모든 사용자

'이탈 위험'과 '휴면'은 **규칙 기반 분류명**이며, 실제 이탈 여부를 검증하거나 예측한 결과는 아닙니다.

In [ ]:
core = (rja_df['R_score'] >= 3) & (rja_df['J_score'] >= 3) & (rja_df['A_score'] >= 3)
active = (rja_df['R_score'] >= 3) & (rja_df['J_score'] >= 3) & (rja_df['A_score'] < 3)
at_risk = (rja_df['R_score'] >= 2) & (rja_df['J_score'] >= 2)

rja_df['Segment'] = np.select(
    [core, active, at_risk],
    ['핵심 구직자', '적극 탐색 구직자', '이탈 위험 구직자'],
    default='휴면 구직자'
)

segment_counts = rja_df['Segment'].value_counts()
print(segment_counts)
assert segment_counts.sum() == len(rja_df)

## 06. 세그먼트별 행동 특성

In [ ]:
segment_analysis = rja_df.groupby('Segment').agg(
    사용자수=('Segment', 'size'),
    평균_최근활동경과일=('Recency', 'mean'),
    평균_공고상세조회수=('job_view', 'mean'),
    평균_지원횟수=('application_count', 'mean')
)
segment_analysis['사용자비율(%)'] = segment_analysis['사용자수'] / len(rja_df) * 100

display(segment_analysis.round(2))

## 07. 세그먼트 분포 시각화

In [ ]:
segment_counts.reindex(
    ['핵심 구직자', '적극 탐색 구직자', '이탈 위험 구직자', '휴면 구직자']
).plot(kind='bar', figsize=(9, 4), rot=0)

plt.title('2023년 구직자 세그먼트 분포')
plt.xlabel('세그먼트')
plt.ylabel('사용자 수')
plt.tight_layout()
plt.show()

## 해석상 주의점

- **적극 탐색 구직자**는 최근 활동 및 상세 조회 점수가 높으나 지원 점수가 상대적으로 낮습니다. **지원하지 않는 이유**까지 확인된 것은 아닙니다.
- 세그먼트 점수 구간은 원본 팀 프로젝트에서 선택한 분석 기준으로, 구간을 변경하면 분류 결과가 달라집니다.
- '이탈 위험 구직자' 그룹에는 일부 지원 경험이 있는 사용자도 포함될 수 있습니다.
- 본 프로젝트는 규칙 기반 세그먼테이션이며, 이탈 확률이나 개인화 추천 효과를 검증한 모델은 아닙니다.
- 개인 식별자(`user_uuid`)가 포함된 원본 데이터 및 실행 결과는 외부에 공개하지 않습니다.